# **Ingest Circuits.csv file**
1. Read the file using spark dataframe reader API
2. Add Metadata Columns 
     .Source Files 
     . Ingestion Time Stamp 
3. Write to bronze delta table      

In [0]:
%run ../00-common/01_Environment-Config

In [0]:
source_file = f"{landing_folder_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"


#### **Step 1 - Read CSV file using the dataframe reader** 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
circuit_schema = StructType([
    StructField('circuitId', StringType(), True), 
    StructField('url', StringType(), True), 
    StructField('circuitName', StringType(), True), 
    StructField('lat', DoubleType(), True), 
    StructField('long', DoubleType(), True),     
    StructField('locality', StringType(), True), 
    StructField('country', StringType(), True)
])


In [0]:
circuits_df = (
spark.read.format('csv') 
    .option('header','True')
  # .option('inferSchema','True')
    .option('mode','FAILFAST')
    .schema(circuit_schema) 
    .load(source_file)
)

In [0]:
display(circuits_df)

### **Step 2 - Add Meta Data Columns**
- Source File
- Ingestions Time Stamp 

In [0]:
from pyspark.sql import functions as F
circuit_final_df = (
        circuits_df
        .withColumn('ingestion_timestamp', F.current_timestamp())
        .withColumn('source_file', F.col('_metadata.file_path'))
)        


In [0]:
display(circuit_final_df)

### **Step 3 : Write to bronze delta table**


In [0]:
(
    circuit_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable('formula1.bronze.circuits')
)

In [0]:
%sql
select * from formula1.bronze.circuits

In [0]:
display(spark.table(table_name))